In [1]:
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
import time


from sklearn.metrics import *
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold
from IPython import display
from copy import deepcopy
from numpy import *
from random import gauss
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
from skopt import gp_minimize
from sklearn.ensemble import AdaBoostClassifier

# warnings.filterwarnings("ignore", category=DeprecationWarning) 
# warnings.filterwarnings('ignore')

#################################################
#------------------ Load Data ------------------#
#################################################

patient_info = pd.read_excel('Dataset.xlsx','patient_information',engine='openpyxl')
cnv_5mb_counts = pd.read_excel('Dataset.xlsx','cnv_5mb_counts',engine='openpyxl').loc[:,['patientID','cnv_Count']].rename(columns={'cnv_Count': 'cnv_Count_5mb'})
cnv_gistic_counts = pd.read_excel('Dataset.xlsx','cnv_gistic_counts',engine='openpyxl').loc[:,['patientID','cnv_Count']].rename(columns={'cnv_Count': 'cnv_Count_gistic'})
snv = pd.read_excel('Dataset.xlsx','snv_feature_matrix',engine='openpyxl')
clinical_info = pd.read_excel('Clinical Feature.xlsx','Supplementary Table 2',engine='openpyxl').drop(['Subject type', 'Subject group', 'Stage (AJCC v7)', 'Pack-years'], axis=1)
clinical_info = clinical_info.rename(columns={'Histology':'Histology label'})

In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

seed = 1105
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)

## Data

In [3]:
##################################################
#---- Data Preprocessing Function defination ---#
################################################

#----- OneHot categorical variable -----#
def OneHotText(x):
    
    """
    Function description: one hot encoding from text to integer and then one_hot_form
    
    """
    LabelTextEncoder = preprocessing.LabelEncoder()
    one_hot = LabelTextEncoder.fit_transform(x).reshape(len(x),1)
    OneHotTextEncoder = preprocessing.OneHotEncoder(sparse=False)
    one_hot = OneHotTextEncoder.fit_transform(one_hot)
    
    return one_hot,LabelTextEncoder,OneHotTextEncoder


#----- Normalization -----#
def MinMaxScale(x):
    
    """
    Function description: implement the minimax normalization
    
    """
    minn = np.min(x)
    maxx = np.max(x)
    x = (x-minn)/(maxx-minn)
    return x

def MinMaxLogScale(x):
    
    """
    Function description: compute the logarithm of all numeric features forst and then implement the minimax normalization
    
    """
    if any([i<0 for i in x]):
        x = [i+abs(min(x)) for i in x]
        
    if any([i==0 for i in x]):
        nozero = min(log10(list(filter(lambda val: abs(val)>=  10**-10, x))))-10
        x = [i+10**nozero for i in x]
    x = log10(x)
    minn = np.min(x)
    maxx = np.max(x)
    x = (x-minn)/(maxx-minn)
    return x

class Normalization:
    def __init__(self, method, **kwargs):
        super(Normalization, self).__init__(**kwargs)
        self.method = method
        
    def normalize(self,x):
        if self.method == 'MinMaxScale':
            return MinMaxScale(x)
        if self.method == 'MinMaxLogScale':
            return MinMaxLogScale(x)
        
def normalizer(x, list_to_normalization, scale):
    for i in list_to_normalization:
        x[i] = scale.normalize(x[i])
    return x


#----- Flatten SNV -----#
def FlattenSNV(snv_multiindex):
    
    """
    Function description: extract and flatten all SNV features into a 2D array
    
    """
    snv_array = []
    for i in range(snv_multiindex.shape[0]):
        temp1 = []
        for j in snv_multiindex.iloc[i]:
            if (type(j) == np.ndarray or type(j) == list):
                for k in j:
                    temp1.append(k)
            else:
                temp1.append(j)
        snv_array.append(temp1)
    snv_array = np.array(snv_array) 
    return snv_array


#----- Truncate and padding SNV -----#
def TruncatePadding(snv_array, max_variats):
    
    """
    Function description: truncate or padding variable-length variants
    
    """
    if len(snv_array) >= max_variats:
        return snv_array[:max_variats]  # truncate
    else:
        return np.append(snv_array,np.zeros((max_variats - snv_array.shape[0],snv_array.shape[1])), axis=0) # padding with zero array

In [4]:
def GroupAndExtractSNV(x1, x_snv, max_variats):
    x = deepcopy(x1)
    num_patients  = len(x)
    snv_group = np.array(x_snv.groupby('patientID'),dtype = object)
        #----- Flatten SNV -----#
    len_flatten_variants = FlattenSNV(snv_group[0][1].drop(columns='patientID')).shape[1] # length of flatten variation
        #----- Truncate and padding SNV -----#
    x['snv'] = [np.zeros((max_variats,len_flatten_variants)) for i in range(num_patients)] # fill snv with 0
    x['snv_valid_len'] = np.zeros(num_patients)
    for i in range(snv_group.shape[0]):
        patient_id = snv_group[i][0]
        snv_multiindex = snv_group[i][1].drop(columns='patientID')
        snv_array =  FlattenSNV(snv_multiindex) # Flatten
        num_repeat = len(x[(x.index == patient_id)])
        
        if num_repeat == 1:
            x.loc[patient_id,'snv_valid_len'] = snv_array.shape[0]
            x.loc[patient_id,'snv'] = [TruncatePadding(snv_array, max_variats)]# Truncate & padding

        else:
            tmp = x.loc[patient_id].iloc[0]
            tmp['snv_valid_len'] = snv_array.shape[0]# Denote valid length
            tmp['snv'] = TruncatePadding(snv_array, max_variats)# Truncate & padding
            x = x.drop([patient_id], axis=0)
            for i in range(num_repeat):
                x = x.append(tmp) 
    return x



def Union(sub_train):
    length = sub_train.iloc[0]['snv'].shape[0]*sub_train.iloc[0]['snv'].shape[1]
    snv1 = zeros((len(sub_train),length))
    for i in range(len(sub_train)):
        snv1[i,:] = sub_train.iloc[i]['snv'].reshape(1,length)
    print(snv1.shape)

    sub_train_x = np.concatenate([snv1, sub_train[['cnv_Count_5mb','cnv_Count_gistic', 'Age (years)', 'Sex', 
                                                   'Smoker', 'Plasma volume used', 'Plasma cfDNA concentration (ng/mL)', 
                                                   'Plasma DNA input (ng)']]], axis=1)
    sub_train_y =  sub_train['label']
    return sub_train_x, sub_train_y

In [5]:
##################################################
#-------------- Data Preprocessing -------------#
################################################

#----- Parameters -----#
num_patients_train = len(patient_info[patient_info.split_info == 'training'])# number of patients in training set
num_patients_test = len(patient_info[patient_info.split_info == 'validation'])# number of patients in training set
max_variats = 84 # max number of variation for each patient
num_var_attri = snv.shape[1] # number of attributes to discribe each variation
scale = Normalization('MinMaxLogScale') # choose normalizer

#----- OneHot categorical variable -----#
one_hot,_,_ = OneHotText(snv['Gene'])
snv['Gene'] = [one_hot[i] for i in range(len(one_hot))]

one_hot,_,_ = OneHotText(snv['Variant.type'])
snv['Variant.type'] = [one_hot[i] for i in range(len(one_hot))]

snv['BaseConversion'] = snv['Tumor.allele']+snv['Ref..allele']
one_hot,_,_ = OneHotText(snv['BaseConversion'])
snv['BaseConversion'] = [one_hot[i] for i in range(len(one_hot))]

one_hot,_,_ = OneHotText(snv['Chr'])
snv['Chr'] = [one_hot[i] for i in range(len(one_hot))]

one_hot,_,_ = OneHotText(snv['germline_reads'])
snv['germline_reads'] = [one_hot[i] for i in range(len(one_hot))]

# drop coded feature
snv = snv.drop(['Tumor.allele', 'Ref..allele', 'variant', 'lookup', 
                'Case', 'cohort', 'A>CUnstranded', 'A>GUnstranded',
                'A>TUnstranded', 'C>AUnstranded', 'C>GUnstranded', 
                'G>AUnstranded', 'pass_basic_QC', 'blacklist1_oncogenes',
                'pass_basic_QC_coding'], axis=1)

# snv feature list to scale
list_to_normalization = ['Position','Percent.mutant.allele', 'variant_count_norm','ontarget_shoulder_tile_mut_count_norm', 
                         'duplex_reads', 'germline_NumNonZeroMeanAF_Pval', 'mean_bc_family_size', 'watsoncrickfisher.p',
                         'mean_total_bc_errors_corrected', 'mean_norm_varpos', 'mean_var_phred_score', 'mean_num_non_ref_bases',
                         'fraction_MapQmin30', 'variant_norm_bc_fam_size', 'normalized_depth','gnomad_maxAF',
                         'size_selected_adjustment','germlinebg_Bayesian_pval', 'cfdnabg_Bayesian_pval',
                         'sum_log10_frag_size_enrich_score', 'germline_depth_at_position', 'cosmic_total_count', 'cosmic_lung_count']

In [6]:
#----- OneHot categorical variable of clinical features ----#
one_hot,_,_ = OneHotText(clinical_info['Sex'])
clinical_info['Sex'] = deepcopy(one_hot)

one_hot,_,_ = OneHotText(clinical_info['Histology label'])
clinical_info['Histology label'] = deepcopy(one_hot)

one_hot,_,_ = OneHotText(clinical_info['Smoker'])
clinical_info['Smoker'] = deepcopy(one_hot)

In [7]:
##################################################
#-------------- Obtain Training Set -------------#
################################################

train_x = patient_info[patient_info.split_info == 'training'].set_index('patientID').\
join(cnv_5mb_counts.set_index('patientID')).join(cnv_gistic_counts.set_index('patientID')).drop('SampleID', axis=1).\
join(clinical_info.set_index('Patient ID'))

#----- cnv normalization -----#
cnv_list_to_normalization = ['cnv_Count_5mb','cnv_Count_gistic']
train_x = normalizer(train_x, cnv_list_to_normalization, scale)
#----- snv normalization -----#
train_snv = deepcopy(snv[['SU2'not in i for i in snv['patientID']]])
train_snv = normalizer(train_snv, list_to_normalization[0:1], scale)
#----- clinical features normalization -----#
clinical_list_to_normalization = ['Age (years)', 'Plasma volume used', 'Plasma cfDNA concentration (ng/mL)', 'Plasma DNA input (ng)']
train_x = normalizer(train_x, clinical_list_to_normalization, scale)
    
#----- Label -----#    
train_x['label'] = [1 if 'LUP' in train_x.index[i] else 0 for i in range(train_x.shape[0])]
one_hot,_,_ = OneHotText(train_x['Stage_group'])
train_x['stage_label'] = [one_hot[i] for i in range(len(one_hot))]
train_x = train_x.drop(['Stage_group'], axis=1)
    
# # x.to_excel("preprocessed_training_dataset.xlsx")     

In [8]:
##################################################
#-------------- Obtain Test Set -------------#
################################################

test_x = patient_info[patient_info.split_info == 'validation'].set_index('patientID').\
join(cnv_5mb_counts.set_index('patientID')).join(cnv_gistic_counts.set_index('patientID')).drop('SampleID', axis=1).\
join(clinical_info.set_index('Patient ID'))

#----- cnv normalization -----#
test_x = normalizer(test_x, cnv_list_to_normalization, scale)
#----- snv normalization -----#
test_snv = deepcopy(snv[['SU2' in i for i in snv['patientID']]])
test_snv = normalizer(test_snv, list_to_normalization, scale)
#----- clinical features normalization -----#
test_x = normalizer(test_x, clinical_list_to_normalization, scale)
    
#----- Label -----#    
test_x['label'] = [1 if 'SU2CS' in test_x.index[i] else 0 for i in range(test_x.shape[0])]
one_hot,_,_ = OneHotText(test_x['Stage_group'])
test_x['stage_label'] = [one_hot[i] for i in range(len(one_hot))]
test_x = test_x.drop(['Stage_group'], axis=1)
    
# # x.to_excel("preprocessed_testing_dataset.xlsx")     

In [9]:
print(train_x.shape)
print(test_x.shape)

(160, 12)
(94, 12)


In [10]:
def Resampler(x,y): 
    random.seed(1105)
    lable_counts = y.value_counts()
    list_to_resample = lable_counts.index.drop(lable_counts.index[argmax(lable_counts)])
    for i in list_to_resample:
        num_resample = max(lable_counts) - lable_counts.loc[i]
        resample_index = random.randint(lable_counts.loc[i], size=num_resample)
        x = x.append(x[x.label == i].iloc[resample_index])
    return x

num_steps = max_variats

sub_train = GroupAndExtractSNV(train_x, train_snv, num_steps)
sub_train = Resampler(sub_train,sub_train['label'])
sub_train_x, sub_train_y =  Union(sub_train)

sub_test =  GroupAndExtractSNV(test_x, test_snv, num_steps)
sub_test_x, sub_test_y =  Union(sub_test)

(208, 22428)
(94, 22428)


In [11]:
from sklearn.ensemble import RandomForestClassifier

test_all = []
for i in range(100):
    clf = RandomForestClassifier(n_estimators=10,random_state=1105+i)
    
    clf.fit(sub_train_x, sub_train_y)
    y_hat = clf.predict(sub_test_x)
    
    auc = roc_auc_score(sub_test_y, clf.predict_proba(sub_test_x)[:, 1])
    acc = accuracy_score(sub_test_y, y_hat)
    test_all.append([auc, acc])
    
test_all = np.array(test_all)
test_auc, test_acc = test_all[:,0], test_all[:,1]
print(f'On test set: Accuracy {test_acc.mean():.3f}, Auc {test_auc.mean():.3f}.')   
#-----  Save performance -----#
df = (('test_acc', 'test_auc'), test_acc, test_auc)
df = pd.DataFrame(df)
file_name = 'RandomForestClassifier.xlsx' 
df.to_excel(file_name,index = False)

On test set: Accuracy 0.609, Auc 0.654.


In [12]:
from sklearn.ensemble import ExtraTreesClassifier

test_all = []
for i in range(100):
    clf = ExtraTreesClassifier(random_state=1105+i)
    
    clf.fit(sub_train_x, sub_train_y)
    y_hat = clf.predict(sub_test_x)
    
    auc = roc_auc_score(sub_test_y, clf.predict_proba(sub_test_x)[:, 1])
    acc = accuracy_score(sub_test_y, y_hat)
    test_all.append([auc, acc])
    
test_all = np.array(test_all)
test_auc, test_acc = test_all[:,0], test_all[:,1]
print(f'On test set: Accuracy {test_acc.mean():.3f}, Auc {test_auc.mean():.3f}.')   
#-----  Save performance -----#
df = (('test_acc', 'test_auc'), test_acc, test_auc)
df = pd.DataFrame(df)
file_name = 'ExtraTreesClassifier.xlsx' 
df.to_excel(file_name,index = False)

On test set: Accuracy 0.625, Auc 0.686.


In [13]:
from sklearn.svm import SVC

test_all = []
for i in range(100):
    clf = SVC(gamma='auto', probability=True, random_state=1105+i)
    
    clf.fit(sub_train_x, sub_train_y)
    y_hat = clf.predict(sub_test_x)
    
    auc = roc_auc_score(sub_test_y, clf.predict_proba(sub_test_x)[:, 1])
    acc = accuracy_score(sub_test_y, y_hat)
    test_all.append([auc, acc])
    
test_all = np.array(test_all)
test_auc, test_acc = test_all[:,0], test_all[:,1]
print(f'On test set: Accuracy {test_acc.mean():.3f}, Auc {test_auc.mean():.3f}.')   
#-----  Save performance -----#
df = (('test_acc', 'test_auc'), test_acc, test_auc)
df = pd.DataFrame(df)
file_name = 'SVC.xlsx' 
df.to_excel(file_name,index = False)

On test set: Accuracy 0.511, Auc 0.707.


In [14]:
from sklearn.linear_model import SGDClassifier
test_all = []
for i in range(100):
    clf = SGDClassifier(loss="log",random_state=1105+i)
    
    clf.fit(sub_train_x, sub_train_y)
    y_hat = clf.predict(sub_test_x)
    
    auc = roc_auc_score(sub_test_y, clf.predict_proba(sub_test_x)[:, 1])
    acc = accuracy_score(sub_test_y, y_hat)
    test_all.append([auc, acc])
    
test_all = np.array(test_all)
test_auc, test_acc = test_all[:,0], test_all[:,1]
print(f'On test set: Accuracy {test_acc.mean():.3f}, Auc {test_auc.mean():.3f}.')   
#-----  Save performance -----#
df = (('test_acc', 'test_auc'), test_acc, test_auc)
df = pd.DataFrame(df)
file_name = 'SGDClassifier.xlsx' 
df.to_excel(file_name,index = False)

On test set: Accuracy 0.610, Auc 0.589.


In [16]:
from sklearn.gaussian_process import GaussianProcessClassifier

test_all = []
for i in range(100):
    clf = GaussianProcessClassifier(random_state=1105+i)
    
    clf.fit(sub_train_x, sub_train_y)
    y_hat = clf.predict(sub_test_x)
    
    auc = roc_auc_score(sub_test_y, clf.predict_proba(sub_test_x)[:, 1])
    acc = accuracy_score(sub_test_y, y_hat)
    
    test_all.append([auc, acc])
    
test_all = np.array(test_all)
test_auc, test_acc = test_all[:,0], test_all[:,1]
print(f'On test set: Accuracy {test_acc.mean():.3f}, Auc {test_auc.mean():.3f}.')   
#-----  Save performance -----#
df = (('test_acc', 'test_auc'), test_acc, test_auc)
df = pd.DataFrame(df)
file_name = 'GaussianProcessClassifier.xlsx' 
df.to_excel(file_name,index = False)

On test set: Accuracy 0.511, Auc 0.691.
